# Semiconductor Revenue-Growth Model — Financial + SEC Filing Text

This notebook runs after the SEC ingestion/feature notebook and compares:

1. **Financial-only logistic regression**
2. **TF-IDF filing-text model**
3. **Pretrained sentence-embedding model**
4. **Combined financial + sentence-embedding model**
5. Optional **FinBERT sentiment features**

All models use the same chronological train/validation/test framework. Model
selection uses validation Brier score, while classification thresholds are
selected using validation balanced accuracy.

The text branch is designed for the writing in 10-Q and 10-K filings,
especially **Management's Discussion and Analysis (MD&A)** and
**Risk Factors**. It can use text already present in the experiment dataset,
merge a separate text parquet/CSV, or download filing HTML from EDGAR when
`cik` and `accession_number` are available.


In [ ]:
# Colab setup
!pip -q install pandas numpy pyarrow scikit-learn joblib matplotlib requests beautifulsoup4 tqdm sentence-transformers transformers

## 1. Configuration

Set `DATA_PATH` directly when you know the parquet/CSV location. When it is `None`, the notebook searches common locations used by the SEC project.

`NEUTRAL_BAND = 0.02` means that next-quarter YoY growth must change by more than **2 percentage points** to receive an acceleration/deceleration label. Observations inside ±2 percentage points are treated as neutral and excluded from the binary model.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

# Set to a Path(...) to override automatic discovery.
DATA_PATH: Path | None = None

# Used only if the loaded dataset does not already contain a split column.
TRAIN_END = pd.Timestamp("2019-12-31")
VALIDATION_END = pd.Timestamp("2021-12-31")

# Two percentage points. Try 0.01 and 0.03 in the sensitivity table as well.
NEUTRAL_BAND = 0.02

C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
THRESHOLD_GRID = np.linspace(0.20, 0.80, 121)

OUTPUT_DIR = Path("/content/semiconductor_model_improved")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

# ---------------------------------------------------------------------
# SEC filing-text configuration
# ---------------------------------------------------------------------

# Optional parquet/CSV containing filing text. Leave None to search common
# paths, use a text column already in the experiment dataset, or download
# from EDGAR.
TEXT_DATA_PATH: Path | None = None

# To download filings directly from SEC EDGAR, replace this placeholder with
# your real name and email. The SEC requires an identifying User-Agent.
SEC_USER_AGENT = "YOUR NAME your.email@example.com"
DOWNLOAD_SEC_TEXT_FROM_EDGAR = True

# Fast sentence-embedding baseline. Change to
# "sentence-transformers/all-mpnet-base-v2" for a larger/slower model.
TEXT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Text processing controls.
MIN_TEXT_CHARS = 500
MAX_DOCUMENT_CHARS = 120_000
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
MAX_CHUNKS_PER_FILING = 16
TEXT_BATCH_SIZE = 32
TEXT_PCA_COMPONENTS = 32

# Optional financial-sentiment features. This is slower and is disabled by
# default. It is a sentiment classifier, not the primary embedding model.
RUN_FINBERT_SENTIMENT = False
FINBERT_MODEL_NAME = "ProsusAI/finbert"

TEXT_CACHE_DIR = OUTPUT_DIR / "sec_text_cache"
TEXT_CACHE_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load the experiment dataset

The code accepts either the parquet produced by the large SEC notebook or the earlier CSV proof-of-concept dataset. It preserves the existing chronological split when available.

In [ ]:
def discover_data_path() -> Path:
    candidates = [
        Path("/content/drive/MyDrive/sec_research_large_150/sec_experiment_large.parquet"),
        Path("/content/drive/MyDrive/sec_research_semiconductor/sec_experiment_semiconductor.parquet"),
        Path("/content/sec_research_large_150/sec_experiment_large.parquet"),
        Path("/content/sec_research_semiconductor/sec_experiment_semiconductor.parquet"),
        Path("sec_experiment_large.parquet"),
        Path("sec_experiment_semiconductor.parquet"),
        Path("model_dataset.csv"),
    ]

    for candidate in candidates:
        if candidate.exists():
            print("Found dataset automatically:", candidate)
            return candidate

    # Colab fallback: ask the user to upload the parquet or CSV.
    try:
        from google.colab import files

        print(
            "Dataset not found automatically.\n"
            "Upload the parquet or CSV created by your SEC data notebook."
        )
        uploaded = files.upload()

        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")

        supported_files = [
            Path(filename)
            for filename in uploaded
            if Path(filename).suffix.lower() in {".parquet", ".csv"}
        ]

        if not supported_files:
            raise FileNotFoundError(
                "The uploaded file must be a .parquet or .csv dataset."
            )

        selected = supported_files[0]
        print("Using uploaded dataset:", selected)
        return selected

    except ImportError as exc:
        raise FileNotFoundError(
            "No experiment dataset was found. Set DATA_PATH to the full "
            "parquet/CSV path, or run this notebook in Colab and upload "
            "the dataset when prompted."
        ) from exc


resolved_path = DATA_PATH if DATA_PATH is not None else discover_data_path()

if not resolved_path.exists():
    raise FileNotFoundError(
        f"DATA_PATH does not exist: {resolved_path}. "
        "Check the filename and folder path."
    )

if resolved_path.suffix.lower() == ".parquet":
    raw = pd.read_parquet(resolved_path)
elif resolved_path.suffix.lower() == ".csv":
    raw = pd.read_csv(resolved_path)
else:
    raise ValueError(
        f"Unsupported file type: {resolved_path.suffix}. "
        "Use a .parquet or .csv file."
    )

print("Loaded:", resolved_path)
print("Rows:", len(raw), "| Columns:", len(raw.columns))
display(pd.DataFrame({"column": raw.columns, "dtype": raw.dtypes.astype(str)}))

## 3. Standardize columns and rebuild the cleaner target

The model predicts whether **next-quarter YoY revenue growth** meaningfully accelerates or decelerates.

\[
\text{growth change}_{i,t+1}
=
g_{i,t+1}-g_{i,t}
\]

\[
y_{i,t} =
\begin{cases}
1 & \text{if growth change} > \delta \\
0 & \text{if growth change} < -\delta \\
\text{neutral} & \text{otherwise}
\end{cases}
\]

Neutral rows are excluded from the binary experiment instead of being forced into a noisy class.

In [ ]:
data = raw.copy()

# Standardize identifiers and dates.
if "cik" not in data.columns:
    if "ticker" not in data.columns:
        raise ValueError("The dataset must contain either cik or ticker.")
    data["cik"] = data["ticker"].astype(str)

if "ticker" not in data.columns:
    data["ticker"] = data["cik"].astype(str)

date_candidates = [
    "quarter_end",
    "feature_cutoff_date",
    "label_available_date",
]
for column in date_candidates:
    if column in data.columns:
        data[column] = pd.to_datetime(data[column], errors="coerce")

if "quarter_end" not in data.columns:
    raise ValueError("The dataset must contain quarter_end.")

data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

# Map names from the earlier proof-of-concept dataset when needed.
rename_aliases = {
    "revenue_mm": "revenue",
    "inventory_mm": "inventory",
    "accounts_receivable_mm": "accounts_receivable",
    "operating_cash_flow_mm": "operating_cash_flow",
    "inventory_ratio": "inventory_to_ttm_revenue",
    "ar_ratio": "receivables_to_ttm_revenue",
    "ocf_margin": "cash_flow_margin",
}
for old_name, new_name in rename_aliases.items():
    if new_name not in data.columns and old_name in data.columns:
        data[new_name] = pd.to_numeric(data[old_name], errors="coerce")

numeric_candidates = [
    "revenue",
    "revenue_yoy_growth",
    "revenue_momentum",
    "next_quarter_growth",
    "gross_margin",
    "operating_margin",
    "cash_flow_margin",
    "inventory",
    "inventory_to_ttm_revenue",
    "accounts_receivable",
    "receivables_to_ttm_revenue",
    "capital_expenditures",
    "capex_to_revenue",
    "total_assets",
    "log_assets",
    "liabilities_to_assets",
]
for column in numeric_candidates:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

# Reconstruct YoY growth if it is missing and revenue is available.
if "revenue_yoy_growth" not in data.columns and "revenue" in data.columns:
    revenue_lag4 = grouped["revenue"].shift(4)
    quarter_lag4 = grouped["quarter_end"].shift(4)
    gap4 = (data["quarter_end"] - quarter_lag4).dt.days
    data["revenue_yoy_growth"] = np.where(
        gap4.between(320, 410),
        data["revenue"] / revenue_lag4 - 1.0,
        np.nan,
    )

# Reconstruct current momentum if missing.
if "revenue_momentum" not in data.columns:
    previous_growth = grouped["revenue_yoy_growth"].shift(1)
    previous_end = grouped["quarter_end"].shift(1)
    gap1 = (data["quarter_end"] - previous_end).dt.days
    data["revenue_momentum"] = np.where(
        gap1.between(60, 125),
        data["revenue_yoy_growth"] - previous_growth,
        np.nan,
    )

# Reconstruct next-quarter growth if missing.
if "next_quarter_growth" not in data.columns:
    next_growth = grouped["revenue_yoy_growth"].shift(-1)
    next_end = grouped["quarter_end"].shift(-1)
    next_gap = (next_end - data["quarter_end"]).dt.days
    data["next_quarter_growth"] = np.where(
        next_gap.between(60, 125),
        next_growth,
        np.nan,
    )

data["future_growth_change"] = (
    data["next_quarter_growth"] - data["revenue_yoy_growth"]
)

data["target_clean"] = pd.Series(
    np.select(
        [
            data["future_growth_change"] > NEUTRAL_BAND,
            data["future_growth_change"] < -NEUTRAL_BAND,
        ],
        [1.0, 0.0],
        default=np.nan,
    ),
    index=data.index,
)

data["target_status"] = np.select(
    [
        data["future_growth_change"] > NEUTRAL_BAND,
        data["future_growth_change"] < -NEUTRAL_BAND,
        data["future_growth_change"].abs() <= NEUTRAL_BAND,
    ],
    ["Accelerating", "Decelerating", "Neutral"],
    default="Unavailable",
)

target_summary = (
    data["target_status"]
    .value_counts(dropna=False)
    .rename_axis("target_status")
    .reset_index(name="rows")
)
target_summary["fraction"] = target_summary["rows"] / len(data)
display(target_summary)

### Neutral-band sensitivity

Use this table to see the trade-off between cleaner labels and sample size. Do **not** choose the band based on test performance.

In [ ]:
sensitivity_rows = []

for band in [0.00, 0.01, 0.02, 0.03, 0.05]:
    target = np.select(
        [
            data["future_growth_change"] > band,
            data["future_growth_change"] < -band,
        ],
        [1.0, 0.0],
        default=np.nan,
    )
    usable = pd.Series(target).dropna()

    sensitivity_rows.append(
        {
            "neutral_band": band,
            "labeled_rows": len(usable),
            "rows_removed_or_unavailable": len(data) - len(usable),
            "acceleration_rate": usable.mean() if len(usable) else np.nan,
            "deceleration_rate": 1.0 - usable.mean() if len(usable) else np.nan,
        }
    )

display(pd.DataFrame(sensitivity_rows))

## 4. Semiconductor-specific and leakage-aware relative features

Relative features use the **previous calendar quarter's cross-sectional semiconductor median**, not the current quarter's full median. This avoids using peer filings that may not have been public at the row's prediction cutoff.

In [ ]:
data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

previous_end = grouped["quarter_end"].shift(1)
gap1 = (data["quarter_end"] - previous_end).dt.days
valid_qoq = gap1.between(60, 125)

quarter_lag4 = grouped["quarter_end"].shift(4)
gap4 = (data["quarter_end"] - quarter_lag4).dt.days
valid_yoy = gap4.between(320, 410)

def add_qoq_change(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(1)
        data[output] = np.where(valid_qoq, data[column] - lagged, np.nan)

def add_yoy_growth(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(4)
        data[output] = np.where(
            valid_yoy & (lagged.abs() > 1e-12),
            data[column] / lagged - 1.0,
            np.nan,
        )

if "revenue" in data.columns:
    revenue_lag1 = grouped["revenue"].shift(1)
    data["sequential_revenue_growth"] = np.where(
        valid_qoq & (revenue_lag1.abs() > 1e-12),
        data["revenue"] / revenue_lag1 - 1.0,
        np.nan,
    )

add_qoq_change("gross_margin", "gross_margin_change")
add_qoq_change("operating_margin", "operating_margin_change")
add_qoq_change("cash_flow_margin", "cash_flow_margin_change")
add_qoq_change(
    "inventory_to_ttm_revenue",
    "inventory_to_ttm_revenue_change",
)
add_qoq_change(
    "receivables_to_ttm_revenue",
    "receivables_to_ttm_revenue_change",
)
add_qoq_change("capex_to_revenue", "capex_to_revenue_change")

add_yoy_growth("inventory", "inventory_yoy_growth")
add_yoy_growth("accounts_receivable", "receivables_yoy_growth")
add_yoy_growth("capital_expenditures", "capex_yoy_growth")

if "inventory_yoy_growth" in data.columns:
    data["inventory_revenue_growth_gap"] = (
        data["inventory_yoy_growth"] - data["revenue_yoy_growth"]
    )

if "receivables_yoy_growth" in data.columns:
    data["receivables_revenue_growth_gap"] = (
        data["receivables_yoy_growth"] - data["revenue_yoy_growth"]
    )

data["calendar_quarter"] = data["quarter_end"].dt.to_period("Q")

relative_base_features = [
    column
    for column in [
        "revenue_yoy_growth",
        "revenue_momentum",
        "sequential_revenue_growth",
        "gross_margin",
        "gross_margin_change",
        "cash_flow_margin",
        "inventory_to_ttm_revenue",
        "inventory_revenue_growth_gap",
        "receivables_to_ttm_revenue",
        "capex_to_revenue",
    ]
    if column in data.columns
]

quarter_medians = (
    data.groupby("calendar_quarter")[relative_base_features]
    .median()
    .sort_index()
)
prior_quarter_medians = quarter_medians.shift(1).add_suffix(
    "_prior_sector_median"
)

data = data.merge(
    prior_quarter_medians,
    left_on="calendar_quarter",
    right_index=True,
    how="left",
)

for feature in relative_base_features:
    median_column = f"{feature}_prior_sector_median"
    data[f"{feature}_relative_to_sector"] = (
        data[feature] - data[median_column]
    )

engineered_features = [
    column
    for column in data.columns
    if (
        column.endswith("_change")
        or column.endswith("_yoy_growth")
        or column.endswith("_growth_gap")
        or column.endswith("_relative_to_sector")
        or column == "sequential_revenue_growth"
    )
]

print("Engineered features:", len(engineered_features))
display(data[["ticker", "quarter_end"] + engineered_features[:12]].head(10))

## 5. Preserve or create chronological splits

The label availability date is used when available. Rows inside the neutral band are removed **after** the chronological assignment.

In [ ]:
# Use an existing split only when it contains all three required groups.
required_splits = {"train", "validation", "test"}

existing_splits = (
    set(data["split"].dropna().astype(str).str.lower().unique())
    if "split" in data.columns
    else set()
)

existing_split_is_usable = required_splits.issubset(existing_splits)

if existing_split_is_usable:
    data["split"] = data["split"].astype(str).str.lower()
    print("Using the existing train/validation/test split.")
else:
    if "split" in data.columns:
        print(
            "The existing split column does not contain all three groups. "
            "Rebuilding the split chronologically."
        )

    data["split"] = pd.NA

    # Use the most conservative available timestamp:
    # when the target became knowable, then the feature cutoff, then quarter end.
    if (
        "label_available_date" in data.columns
        and data["label_available_date"].notna().any()
    ):
        split_date_column = "label_available_date"
    elif (
        "feature_cutoff_date" in data.columns
        and data["feature_cutoff_date"].notna().any()
    ):
        split_date_column = "feature_cutoff_date"
    else:
        split_date_column = "quarter_end"

    eligible_dates = (
        data.loc[data["target_clean"].notna(), split_date_column]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    if len(eligible_dates) < 3:
        raise ValueError(
            "At least three distinct dated periods are required to create "
            "train, validation, and test splits."
        )

    # Automatic chronological 60% / 20% / 20% split by distinct dates.
    train_position = max(0, min(len(eligible_dates) - 3, int(len(eligible_dates) * 0.60) - 1))
    validation_position = max(
        train_position + 1,
        min(len(eligible_dates) - 2, int(len(eligible_dates) * 0.80) - 1),
    )

    automatic_train_end = eligible_dates.iloc[train_position]
    automatic_validation_end = eligible_dates.iloc[validation_position]

    labeled = data["target_clean"].notna()
    split_dates = data[split_date_column]

    data.loc[
        labeled & (split_dates <= automatic_train_end),
        "split",
    ] = "train"

    data.loc[
        labeled
        & (split_dates > automatic_train_end)
        & (split_dates <= automatic_validation_end),
        "split",
    ] = "validation"

    data.loc[
        labeled & (split_dates > automatic_validation_end),
        "split",
    ] = "test"

    print("Split date column:", split_date_column)
    print("Automatic train end:", automatic_train_end)
    print("Automatic validation end:", automatic_validation_end)

model_data = data[
    data["target_clean"].notna()
    & data["split"].isin(["train", "validation", "test"])
].copy()
model_data["target_clean"] = model_data["target_clean"].astype(int)

split_summary = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        companies=("cik", "nunique"),
        first_quarter=("quarter_end", "min"),
        last_quarter=("quarter_end", "max"),
        acceleration_rate=("target_clean", "mean"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
display(split_summary)

available_splits = set(model_data["split"].dropna().unique())
missing_splits = required_splits - available_splits
if missing_splits:
    raise ValueError(
        f"Could not create these splits: {sorted(missing_splits)}. "
        "The dataset may have too few labeled dates after applying the "
        "neutral band."
    )

for split_name in ["train", "validation", "test"]:
    split_frame = model_data[model_data["split"] == split_name]
    if split_frame["target_clean"].nunique() < 2:
        print(
            f"Warning: {split_name} contains only one target class after "
            f"applying the {NEUTRAL_BAND:.1%} neutral band. "
            "Try reducing NEUTRAL_BAND to 0.01 if model fitting fails."
        )

## 6. Attach SEC filing writing to each observation

The code uses the first available source:

1. Text columns already present in the experiment dataset.
2. A separate parquet/CSV specified by `TEXT_DATA_PATH`.
3. Direct EDGAR downloads using `cik` and `accession_number`.

For direct EDGAR access, edit `SEC_USER_AGENT` with your name and email.
Downloads are cached and deliberately rate-limited. The extracted document
focuses on MD&A and Risk Factors, with cleaned full filing text as a fallback.

The text must correspond to information available by the row's
`feature_cutoff_date`; a leakage check removes text whose filing date is later
than that cutoff.

In [ ]:
import hashlib
import html as html_lib
import json
import re
import time
from collections.abc import Iterable

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


TEXT_COLUMN_CANDIDATES = [
    "sec_text",
    "filing_text",
    "document_text",
    "mda_text",
    "management_discussion_text",
    "risk_factors_text",
    "risk_text",
]


def normalize_accession(value: object) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def combine_available_text_columns(frame: pd.DataFrame) -> pd.Series:
    available = [
        column for column in TEXT_COLUMN_CANDIDATES
        if column in frame.columns
    ]
    if not available:
        return pd.Series("", index=frame.index, dtype="object")

    combined = (
        frame[available]
        .fillna("")
        .astype(str)
        .apply(
            lambda row: "\n\n".join(
                value.strip()
                for value in row
                if value and value.strip()
            ),
            axis=1,
        )
    )
    return combined


def discover_text_data_path() -> Path | None:
    candidates = [
        Path("/content/drive/MyDrive/sec_research_large_150/sec_filing_text.parquet"),
        Path("/content/drive/MyDrive/sec_research_semiconductor/sec_filing_text.parquet"),
        Path("/content/sec_research_large_150/sec_filing_text.parquet"),
        Path("/content/sec_research_semiconductor/sec_filing_text.parquet"),
        Path("sec_filing_text.parquet"),
        Path("sec_filing_text.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError("Text dataset must be a .parquet or .csv file.")


def merge_external_text(
    base: pd.DataFrame,
    text_frame: pd.DataFrame,
) -> pd.DataFrame:
    external = text_frame.copy()

    for frame in [base, external]:
        if "quarter_end" in frame.columns:
            frame["quarter_end"] = pd.to_datetime(
                frame["quarter_end"], errors="coerce"
            )
        if "filing_date" in frame.columns:
            frame["filing_date"] = pd.to_datetime(
                frame["filing_date"], errors="coerce"
            )
        if "accession_number" in frame.columns:
            frame["accession_number"] = frame[
                "accession_number"
            ].map(normalize_accession)
        if "cik" in frame.columns:
            frame["cik"] = frame["cik"].astype(str)
        if "ticker" in frame.columns:
            frame["ticker"] = frame["ticker"].astype(str)

    external["external_sec_text"] = combine_available_text_columns(
        external
    )

    if "accession_number" in base.columns and "accession_number" in external.columns:
        merge_keys = ["accession_number"]
    elif all(
        column in base.columns and column in external.columns
        for column in ["cik", "quarter_end"]
    ):
        merge_keys = ["cik", "quarter_end"]
    elif all(
        column in base.columns and column in external.columns
        for column in ["ticker", "quarter_end"]
    ):
        merge_keys = ["ticker", "quarter_end"]
    else:
        raise ValueError(
            "The text dataset must share accession_number, "
            "(cik, quarter_end), or (ticker, quarter_end) with the "
            "experiment dataset."
        )

    keep_columns = merge_keys + ["external_sec_text"]
    if "filing_date" in external.columns:
        keep_columns.append("filing_date")

    external = (
        external[keep_columns]
        .drop_duplicates(subset=merge_keys, keep="last")
    )

    merged = base.merge(
        external,
        on=merge_keys,
        how="left",
        suffixes=("", "_text_source"),
    )

    merged["sec_text"] = np.where(
        merged["sec_text"].fillna("").str.len()
        >= merged["external_sec_text"].fillna("").str.len(),
        merged["sec_text"].fillna(""),
        merged["external_sec_text"].fillna(""),
    )
    return merged


def clean_filing_html(raw_html: str) -> str:
    soup = BeautifulSoup(raw_html, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    # The project is using the narrative writing; large XBRL tables add
    # many repeated numbers and labels without much prose.
    for table in soup.find_all("table"):
        table.decompose()

    text = soup.get_text(" ")
    text = html_lib.unescape(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_longest_section(
    text: str,
    start_patterns: list[str],
    end_patterns: list[str],
    minimum_chars: int = 500,
    maximum_chars: int = 100_000,
) -> str:
    starts = []
    for pattern in start_patterns:
        starts.extend(re.finditer(pattern, text, flags=re.I))

    ends = []
    for pattern in end_patterns:
        ends.extend(re.finditer(pattern, text, flags=re.I))

    candidates: list[str] = []
    for start in starts:
        possible_ends = [
            end for end in ends
            if end.start() > start.end() + minimum_chars
        ]
        if not possible_ends:
            continue
        end = min(possible_ends, key=lambda match: match.start())
        candidate = text[start.start():end.start()].strip()
        if minimum_chars <= len(candidate) <= maximum_chars:
            candidates.append(candidate)

    return max(candidates, key=len) if candidates else ""


def extract_narrative_sections(clean_text: str) -> str:
    mda = extract_longest_section(
        clean_text,
        start_patterns=[
            r"\bitem\s+7[\.\:\-\s]+management[’']?s?\s+discussion",
            r"\bitem\s+2[\.\:\-\s]+management[’']?s?\s+discussion",
        ],
        end_patterns=[
            r"\bitem\s+7a[\.\:\-\s]+",
            r"\bitem\s+8[\.\:\-\s]+financial",
            r"\bitem\s+3[\.\:\-\s]+quantitative",
            r"\bitem\s+4[\.\:\-\s]+controls",
        ],
    )

    risks = extract_longest_section(
        clean_text,
        start_patterns=[
            r"\bitem\s+1a[\.\:\-\s]+risk\s+factors",
        ],
        end_patterns=[
            r"\bitem\s+1b[\.\:\-\s]+",
            r"\bitem\s+1c[\.\:\-\s]+",
            r"\bitem\s+2[\.\:\-\s]+",
        ],
    )

    sections = []
    if mda:
        sections.append("MANAGEMENT DISCUSSION AND ANALYSIS\n" + mda)
    if risks:
        sections.append("RISK FACTORS\n" + risks)

    if sections:
        return "\n\n".join(sections)[:MAX_DOCUMENT_CHARS]

    # Fallback when filing headings differ from the standard patterns.
    return clean_text[:MAX_DOCUMENT_CHARS]


def valid_sec_user_agent(user_agent: str) -> bool:
    lowered = user_agent.lower()
    return (
        "your name" not in lowered
        and "example.com" not in lowered
        and "@" in user_agent
        and len(user_agent.strip()) >= 8
    )


class EdgarTextDownloader:
    def __init__(self, user_agent: str, cache_dir: Path):
        if not valid_sec_user_agent(user_agent):
            raise ValueError(
                "Replace SEC_USER_AGENT with your real name and email "
                "before downloading from EDGAR."
            )

        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": user_agent,
                "Accept-Encoding": "gzip, deflate",
            }
        )
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.submission_cache: dict[str, dict[str, str]] = {}

    def get_json(self, url: str) -> dict:
        response = self.session.get(url, timeout=60)
        response.raise_for_status()
        time.sleep(0.20)
        return response.json()

    def get_text(self, url: str) -> str:
        response = self.session.get(url, timeout=90)
        response.raise_for_status()
        time.sleep(0.20)
        return response.text

    def _add_submission_rows(
        self,
        mapping: dict[str, str],
        payload: dict,
    ) -> None:
        accessions = payload.get("accessionNumber", [])
        primary_documents = payload.get("primaryDocument", [])
        for accession, document in zip(
            accessions, primary_documents, strict=False
        ):
            if accession and document:
                mapping[str(accession)] = str(document)

    def submission_map(self, cik: str) -> dict[str, str]:
        cik_key = str(int(float(cik))).zfill(10)
        if cik_key in self.submission_cache:
            return self.submission_cache[cik_key]

        payload = self.get_json(
            f"https://data.sec.gov/submissions/CIK{cik_key}.json"
        )
        mapping: dict[str, str] = {}
        self._add_submission_rows(
            mapping,
            payload.get("filings", {}).get("recent", {}),
        )

        # Older filings can be stored in additional submission JSON files.
        for file_info in payload.get("filings", {}).get("files", []):
            file_name = file_info.get("name")
            if not file_name:
                continue
            old_payload = self.get_json(
                f"https://data.sec.gov/submissions/{file_name}"
            )
            self._add_submission_rows(mapping, old_payload)

        self.submission_cache[cik_key] = mapping
        return mapping

    def primary_document(
        self,
        cik: str,
        accession: str,
        supplied_document: object = None,
    ) -> str:
        if supplied_document is not None and not pd.isna(supplied_document):
            supplied = str(supplied_document).strip()
            if supplied:
                return supplied

        mapping = self.submission_map(cik)
        document = mapping.get(accession)
        if document:
            return document

        # Final fallback: choose the largest non-index HTML document.
        cik_number = str(int(float(cik)))
        accession_compact = accession.replace("-", "")
        index_payload = self.get_json(
            "https://www.sec.gov/Archives/edgar/data/"
            f"{cik_number}/{accession_compact}/index.json"
        )
        items = (
            index_payload.get("directory", {}).get("item", [])
        )
        html_items = [
            item for item in items
            if str(item.get("name", "")).lower().endswith(
                (".htm", ".html")
            )
            and "-index." not in str(item.get("name", "")).lower()
            and "filingsummary" not in str(item.get("name", "")).lower()
        ]
        if not html_items:
            raise FileNotFoundError(
                f"No primary HTML document found for {accession}."
            )
        chosen = max(
            html_items,
            key=lambda item: int(item.get("size", 0) or 0),
        )
        return str(chosen["name"])

    def filing_text(
        self,
        cik: str,
        accession: str,
        supplied_document: object = None,
    ) -> str:
        accession = normalize_accession(accession)
        if accession is None:
            return ""

        cache_path = self.cache_dir / f"{accession}.txt"
        if cache_path.exists():
            return cache_path.read_text(
                encoding="utf-8", errors="ignore"
            )

        document = self.primary_document(
            cik, accession, supplied_document
        )
        cik_number = str(int(float(cik)))
        accession_compact = accession.replace("-", "")
        url = (
            "https://www.sec.gov/Archives/edgar/data/"
            f"{cik_number}/{accession_compact}/{document}"
        )
        raw_html = self.get_text(url)
        cleaned = clean_filing_html(raw_html)
        narrative = extract_narrative_sections(cleaned)
        cache_path.write_text(narrative, encoding="utf-8")
        return narrative


# Begin with any text already present in the experiment dataset.
model_data = model_data.copy()
model_data["sec_text"] = combine_available_text_columns(model_data)

# Merge a separate text table when configured or automatically found.
resolved_text_path = (
    TEXT_DATA_PATH
    if TEXT_DATA_PATH is not None
    else discover_text_data_path()
)
if resolved_text_path is not None:
    if not resolved_text_path.exists():
        raise FileNotFoundError(
            f"TEXT_DATA_PATH does not exist: {resolved_text_path}"
        )
    external_text = read_table(resolved_text_path)
    model_data = merge_external_text(model_data, external_text)
    print("Merged SEC text dataset:", resolved_text_path)

# Download only missing filing text, using one request per unique filing.
missing_text = model_data["sec_text"].fillna("").str.len() < MIN_TEXT_CHARS
can_download = (
    DOWNLOAD_SEC_TEXT_FROM_EDGAR
    and missing_text.any()
    and {"cik", "accession_number"}.issubset(model_data.columns)
    and valid_sec_user_agent(SEC_USER_AGENT)
)

if can_download:
    downloader = EdgarTextDownloader(
        SEC_USER_AGENT,
        TEXT_CACHE_DIR / "filings",
    )

    unique_filings = (
        model_data.loc[
            missing_text,
            [
                column
                for column in [
                    "cik",
                    "accession_number",
                    "primary_document",
                ]
                if column in model_data.columns
            ],
        ]
        .dropna(subset=["cik", "accession_number"])
        .drop_duplicates(subset=["cik", "accession_number"])
    )

    downloaded_text: dict[tuple[str, str], str] = {}
    failures = []

    for _, filing in tqdm(
        unique_filings.iterrows(),
        total=len(unique_filings),
        desc="Downloading SEC filings",
    ):
        cik = str(filing["cik"])
        accession = normalize_accession(filing["accession_number"])
        if accession is None:
            continue
        supplied_document = (
            filing.get("primary_document")
            if "primary_document" in filing.index
            else None
        )
        try:
            downloaded_text[(cik, accession)] = (
                downloader.filing_text(
                    cik,
                    accession,
                    supplied_document,
                )
            )
        except Exception as exc:
            failures.append(
                {
                    "cik": cik,
                    "accession_number": accession,
                    "error": str(exc),
                }
            )

    row_keys = list(
        zip(
            model_data["cik"].astype(str),
            model_data["accession_number"].map(normalize_accession),
        )
    )
    downloaded_series = pd.Series(
        [
            downloaded_text.get(key, "")
            for key in row_keys
        ],
        index=model_data.index,
    )
    replace_mask = (
        model_data["sec_text"].fillna("").str.len()
        < downloaded_series.str.len()
    )
    model_data.loc[replace_mask, "sec_text"] = downloaded_series[
        replace_mask
    ]

    if failures:
        failure_frame = pd.DataFrame(failures)
        failure_frame.to_csv(
            TEXT_CACHE_DIR / "edgar_download_failures.csv",
            index=False,
        )
        print(
            f"{len(failures)} filing downloads failed. Details were saved "
            "to edgar_download_failures.csv."
        )

elif missing_text.any():
    print(
        "SEC text is not yet available for all rows.\n"
        "Use one of these options:\n"
        "  1. Put sec_text, filing_text, mda_text, or risk_factors_text "
        "in the experiment dataset.\n"
        "  2. Set TEXT_DATA_PATH to a matching text parquet/CSV.\n"
        "  3. Replace SEC_USER_AGENT with your real name and email so "
        "the notebook can download filings from EDGAR."
    )

# Prevent obvious timing leakage when both timestamps are available.
if (
    "filing_date" in model_data.columns
    and "feature_cutoff_date" in model_data.columns
):
    filing_dates = pd.to_datetime(
        model_data["filing_date"], errors="coerce"
    )
    cutoff_dates = pd.to_datetime(
        model_data["feature_cutoff_date"], errors="coerce"
    )
    late_text = (
        filing_dates.notna()
        & cutoff_dates.notna()
        & (filing_dates > cutoff_dates)
    )
    if late_text.any():
        print(
            f"Removed text from {late_text.sum()} rows because the filing "
            "date was after the feature cutoff date."
        )
        model_data.loc[late_text, "sec_text"] = ""

model_data["text_characters"] = (
    model_data["sec_text"].fillna("").str.len()
)
model_data["has_sec_text"] = (
    model_data["text_characters"] >= MIN_TEXT_CHARS
)

text_coverage = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        rows_with_text=("has_sec_text", "sum"),
        median_text_characters=("text_characters", "median"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
text_coverage["coverage"] = (
    text_coverage["rows_with_text"] / text_coverage["rows"]
)
display(text_coverage)

### What the two text representations capture

- **TF-IDF** learns which words and short phrases are associated with later
  acceleration or deceleration. It is transparent and is the required text
  baseline.
- **Sentence embeddings** map filing chunks into dense vectors, allowing
  semantically similar writing to be close even when the exact vocabulary
  differs.
- **FinBERT sentiment** is optional. It adds positive, neutral, and negative
  tone summaries, but sentiment alone should not replace the broader semantic
  embedding.

## 6. Train compact L2 logistic models

Two versions are compared:

- **Unweighted logistic regression**: usually better calibrated when class imbalance is modest.
- **Balanced logistic regression**: gives each class equal total weight and can improve minority-class recall.

The company identifier is intentionally **not** included in the primary model, preventing the model from simply memorizing company history.

In [ ]:
candidate_numeric_features = [
    "revenue_yoy_growth",
    "revenue_momentum",
    "sequential_revenue_growth",
    "gross_margin",
    "gross_margin_change",
    "operating_margin",
    "operating_margin_change",
    "cash_flow_margin",
    "cash_flow_margin_change",
    "inventory_to_ttm_revenue",
    "inventory_to_ttm_revenue_change",
    "inventory_yoy_growth",
    "inventory_revenue_growth_gap",
    "receivables_to_ttm_revenue",
    "receivables_to_ttm_revenue_change",
    "receivables_yoy_growth",
    "receivables_revenue_growth_gap",
    "capex_to_revenue",
    "capex_to_revenue_change",
    "capex_yoy_growth",
    "log_assets",
    "liabilities_to_assets",
]

candidate_numeric_features += [
    f"{feature}_relative_to_sector"
    for feature in relative_base_features
]

numeric_features = [
    column
    for column in dict.fromkeys(candidate_numeric_features)
    if column in model_data.columns
    and model_data[column].notna().sum() >= 10
    and model_data[column].nunique(dropna=True) > 1
]

categorical_features = [
    column
    for column in ["fiscal_quarter"]
    if column in model_data.columns
]

feature_columns = numeric_features + categorical_features

if not feature_columns:
    raise ValueError("No usable model features were found.")

print("Numeric features:", len(numeric_features))
print("Categorical features:", categorical_features)
display(pd.DataFrame({"feature": feature_columns}))

train_frame = model_data[model_data["split"] == "train"].copy()
validation_frame = model_data[model_data["split"] == "validation"].copy()
test_frame = model_data[model_data["split"] == "test"].copy()

def make_pipeline(C: float, class_weight: str | None) -> Pipeline:
    transformers = []

    if numeric_features:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            )
        )

    if categorical_features:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(strategy="most_frequent"),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical_features,
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

    return Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    C=C,
                    class_weight=class_weight,
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def expected_calibration_error(
    y_true: pd.Series | np.ndarray,
    probabilities: np.ndarray,
    bins: int = 10,
) -> float:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(probabilities, dtype=float)
    edges = np.linspace(0.0, 1.0, bins + 1)
    bin_ids = np.digitize(p, edges[1:-1], right=True)

    ece = 0.0
    for bin_id in range(bins):
        mask = bin_ids == bin_id
        if not mask.any():
            continue
        ece += mask.mean() * abs(y[mask].mean() - p[mask].mean())
    return float(ece)


def safe_auc(y_true: pd.Series, probabilities: np.ndarray) -> float:
    return (
        float(roc_auc_score(y_true, probabilities))
        if y_true.nunique() == 2
        else np.nan
    )


def safe_average_precision(
    y_true: pd.Series,
    probabilities: np.ndarray,
) -> float:
    return (
        float(average_precision_score(y_true, probabilities))
        if y_true.nunique() == 2
        else np.nan
    )


model_variants = {
    "L2 logistic": None,
    "L2 logistic balanced": "balanced",
}

selection_rows: list[dict[str, Any]] = []
selected_models: dict[str, dict[str, Any]] = {}

for model_name, class_weight in model_variants.items():
    best_candidate: dict[str, Any] | None = None

    for C in C_GRID:
        pipeline = make_pipeline(C=C, class_weight=class_weight)
        pipeline.fit(
            train_frame[feature_columns],
            train_frame["target_clean"],
        )
        validation_probability = pipeline.predict_proba(
            validation_frame[feature_columns]
        )[:, 1]

        validation_brier = brier_score_loss(
            validation_frame["target_clean"],
            validation_probability,
        )

        candidate = {
            "model": model_name,
            "C": C,
            "class_weight": class_weight,
            "pipeline": pipeline,
            "validation_probability": validation_probability,
            "validation_brier": validation_brier,
        }

        if (
            best_candidate is None
            or candidate["validation_brier"]
            < best_candidate["validation_brier"]
        ):
            best_candidate = candidate

    assert best_candidate is not None

    threshold_scores = [
        balanced_accuracy_score(
            validation_frame["target_clean"],
            (
                best_candidate["validation_probability"] >= threshold
            ).astype(int),
        )
        for threshold in THRESHOLD_GRID
    ]
    best_threshold = float(
        THRESHOLD_GRID[int(np.argmax(threshold_scores))]
    )

    best_candidate["threshold"] = best_threshold
    best_candidate["validation_balanced_accuracy"] = float(
        max(threshold_scores)
    )
    selected_models[model_name] = best_candidate

    selection_rows.append(
        {
            "model": model_name,
            "selected_C": best_candidate["C"],
            "class_weight": str(class_weight),
            "validation_brier": best_candidate["validation_brier"],
            "validation_balanced_accuracy": best_candidate[
                "validation_balanced_accuracy"
            ],
            "validation_threshold": best_threshold,
        }
    )

selection_table = pd.DataFrame(selection_rows).sort_values(
    ["validation_brier", "validation_balanced_accuracy"],
    ascending=[True, False],
)
display(selection_table)

## 7. Final untouched-test evaluation

Hyperparameters and thresholds come only from train/validation. The selected model is then refit on train + validation and evaluated once on the test period.

In [ ]:
def metric_row(
    model_name: str,
    y_true: pd.Series,
    probabilities: np.ndarray,
    predictions: np.ndarray,
    baseline_probability: float,
) -> dict[str, Any]:
    model_brier = brier_score_loss(y_true, probabilities)
    baseline_probabilities = np.full(len(y_true), baseline_probability)
    baseline_brier = brier_score_loss(y_true, baseline_probabilities)

    return {
        "model": model_name,
        "rows": len(y_true),
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "macro_f1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "accelerating_precision": precision_score(
            y_true,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "accelerating_recall": recall_score(
            y_true,
            predictions,
            pos_label=1,
            zero_division=0,
        ),
        "decelerating_recall": recall_score(
            y_true,
            predictions,
            pos_label=0,
            zero_division=0,
        ),
        "brier_score": model_brier,
        "brier_skill_score": (
            1.0 - model_brier / baseline_brier
            if baseline_brier > 0
            else np.nan
        ),
        "log_loss": log_loss(y_true, probabilities, labels=[0, 1]),
        "roc_auc": safe_auc(y_true, probabilities),
        "average_precision": safe_average_precision(
            y_true, probabilities
        ),
        "ece_10_bins": expected_calibration_error(
            y_true, probabilities, bins=10
        ),
    }


train_validation = pd.concat(
    [train_frame, validation_frame],
    ignore_index=True,
)
y_test = test_frame["target_clean"]
training_prevalence = float(
    train_validation["target_clean"].mean()
)
majority_class = int(training_prevalence >= 0.5)

test_rows: list[dict[str, Any]] = []
prediction_output = test_frame[
    [
        column
        for column in [
            "cik",
            "ticker",
            "company_name",
            "quarter_end",
            "feature_cutoff_date",
            "label_available_date",
            "future_growth_change",
            "target_clean",
        ]
        if column in test_frame.columns
    ]
].copy()

final_models: dict[str, Pipeline] = {}

# Proper baselines.
prior_probabilities = np.full(len(test_frame), training_prevalence)
prior_predictions = (
    prior_probabilities >= 0.5
).astype(int)
test_rows.append(
    metric_row(
        "Prior-probability baseline",
        y_test,
        prior_probabilities,
        prior_predictions,
        baseline_probability=training_prevalence,
    )
)

majority_probabilities = np.full(
    len(test_frame),
    float(majority_class),
)
majority_predictions = np.full(
    len(test_frame),
    majority_class,
)
test_rows.append(
    metric_row(
        "Majority-class baseline",
        y_test,
        majority_probabilities,
        majority_predictions,
        baseline_probability=training_prevalence,
    )
)

for model_name, selected in selected_models.items():
    final_pipeline = make_pipeline(
        C=float(selected["C"]),
        class_weight=selected["class_weight"],
    )
    final_pipeline.fit(
        train_validation[feature_columns],
        train_validation["target_clean"],
    )

    test_probability = final_pipeline.predict_proba(
        test_frame[feature_columns]
    )[:, 1]
    test_prediction = (
        test_probability >= float(selected["threshold"])
    ).astype(int)

    final_models[model_name] = final_pipeline
    prediction_output[
        f"{model_name}_probability"
    ] = test_probability
    prediction_output[
        f"{model_name}_prediction"
    ] = test_prediction

    row = metric_row(
        model_name,
        y_test,
        test_probability,
        test_prediction,
        baseline_probability=training_prevalence,
    )
    row["selected_C"] = selected["C"]
    row["threshold"] = selected["threshold"]
    test_rows.append(row)

test_results = pd.DataFrame(test_rows).sort_values(
    ["brier_score", "balanced_accuracy"],
    ascending=[True, False],
)
display(test_results)

best_model_name = selection_table.iloc[0]["model"]
best_model = final_models[best_model_name]
best_threshold = float(
    selected_models[best_model_name]["threshold"]
)

print("Validation-selected model:", best_model_name)
print("Selected threshold:", best_threshold)

best_predictions = prediction_output[
    f"{best_model_name}_prediction"
].to_numpy()
matrix = confusion_matrix(
    y_test,
    best_predictions,
    labels=[0, 1],
)
confusion_table = pd.DataFrame(
    matrix,
    index=["Actual decelerating", "Actual accelerating"],
    columns=["Predicted decelerating", "Predicted accelerating"],
)
display(confusion_table)

## 9. Text-only and combined model evaluation

Every model below is trained and evaluated on the **same rows with available
SEC text**, making the comparison fair.

The TF-IDF vocabulary and the sentence-embedding PCA transformation are fitted
only on training data. Pretrained embeddings are computed for all documents,
but no target labels are used when creating them.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import clone


text_model_data = model_data[
    model_data["has_sec_text"]
    & model_data["split"].isin(["train", "validation", "test"])
].copy()

text_split_counts = (
    text_model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        companies=("cik", "nunique"),
        classes=("target_clean", "nunique"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
display(text_split_counts)

TEXT_MODELS_READY = True
for required_split in ["train", "validation", "test"]:
    subset = text_model_data[
        text_model_data["split"] == required_split
    ]
    if subset.empty:
        print(
            f"Text model skipped: no text rows are available in "
            f"{required_split}."
        )
        TEXT_MODELS_READY = False
    elif subset["target_clean"].nunique() < 2:
        print(
            f"Text model skipped: {required_split} has only one target "
            "class after filtering for text."
        )
        TEXT_MODELS_READY = False

if not TEXT_MODELS_READY:
    print(
        "Add more filing text, reduce NEUTRAL_BAND to 0.01, or expand the "
        "company/date universe before running the text comparison."
    )

In [ ]:
def chunk_document(
    text: str,
    chunk_words: int = CHUNK_WORDS,
    overlap_words: int = CHUNK_OVERLAP_WORDS,
    maximum_chunks: int = MAX_CHUNKS_PER_FILING,
) -> list[str]:
    words = str(text).split()
    if not words:
        return []

    step = max(1, chunk_words - overlap_words)
    chunks = [
        " ".join(words[start:start + chunk_words])
        for start in range(0, len(words), step)
        if len(words[start:start + chunk_words]) >= 30
    ]

    if not chunks:
        return [" ".join(words)]

    if len(chunks) > maximum_chunks:
        selected_indices = np.linspace(
            0,
            len(chunks) - 1,
            maximum_chunks,
            dtype=int,
        )
        chunks = [chunks[index] for index in selected_indices]

    return chunks


def text_hash(text: str) -> str:
    payload = (
        TEXT_EMBEDDING_MODEL
        + "\n"
        + str(MAX_CHUNKS_PER_FILING)
        + "\n"
        + str(text)
    )
    return hashlib.sha1(
        payload.encode("utf-8", errors="ignore")
    ).hexdigest()


def build_sentence_embeddings(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    from sentence_transformers import SentenceTransformer

    working = frame.copy()
    working["text_hash"] = working["sec_text"].map(text_hash)

    safe_model_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        TEXT_EMBEDDING_MODEL,
    )
    cache_path = (
        TEXT_CACHE_DIR
        / f"document_embeddings_{safe_model_name}.parquet"
    )

    cached = pd.DataFrame()
    if cache_path.exists():
        cached = pd.read_parquet(cache_path)

    cached_hashes = (
        set(cached["text_hash"])
        if not cached.empty and "text_hash" in cached.columns
        else set()
    )

    unique_documents = (
        working[["text_hash", "sec_text"]]
        .drop_duplicates("text_hash")
    )
    missing_documents = unique_documents[
        ~unique_documents["text_hash"].isin(cached_hashes)
    ]

    if not missing_documents.empty:
        encoder = SentenceTransformer(TEXT_EMBEDDING_MODEL)

        flat_chunks: list[str] = []
        owners: list[str] = []
        for row in missing_documents.itertuples(index=False):
            chunks = chunk_document(row.sec_text)
            flat_chunks.extend(chunks)
            owners.extend([row.text_hash] * len(chunks))

        if not flat_chunks:
            raise ValueError("No valid text chunks were produced.")

        chunk_embeddings = encoder.encode(
            flat_chunks,
            batch_size=TEXT_BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        chunk_frame = pd.DataFrame(chunk_embeddings)
        chunk_frame.insert(0, "text_hash", owners)

        document_embeddings = (
            chunk_frame.groupby("text_hash", sort=False)
            .mean()
            .reset_index()
        )

        embedding_columns = [
            column
            for column in document_embeddings.columns
            if column != "text_hash"
        ]
        matrix = document_embeddings[embedding_columns].to_numpy()
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        matrix = matrix / np.maximum(norms, 1e-12)
        document_embeddings[embedding_columns] = matrix

        if cached.empty:
            cached = document_embeddings
        else:
            cached = pd.concat(
                [cached, document_embeddings],
                ignore_index=True,
            ).drop_duplicates("text_hash", keep="last")

        cached.to_parquet(cache_path, index=False)

    embedding_columns_in_cache = [
        column for column in cached.columns
        if column != "text_hash"
    ]
    renamed = {
        column: f"text_embedding_{int(column):03d}"
        for column in embedding_columns_in_cache
    }
    cached = cached.rename(columns=renamed)
    embedding_columns = list(renamed.values())

    working = working.merge(
        cached,
        on="text_hash",
        how="left",
    )
    return working, embedding_columns


if TEXT_MODELS_READY:
    text_model_data, text_embedding_columns = (
        build_sentence_embeddings(text_model_data)
    )
    print(
        "Sentence embedding dimensions:",
        len(text_embedding_columns),
    )
else:
    text_embedding_columns = []

In [ ]:
def build_finbert_sentiment_features(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    import torch
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
    )

    output = frame.copy()
    feature_names = [
        "finbert_positive_mean",
        "finbert_neutral_mean",
        "finbert_negative_mean",
        "finbert_negative_max",
        "finbert_negative_std",
        "finbert_positive_minus_negative",
    ]

    if not RUN_FINBERT_SENTIMENT:
        for feature in feature_names:
            output[feature] = 0.0
        return output, []

    tokenizer = AutoTokenizer.from_pretrained(
        FINBERT_MODEL_NAME
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        FINBERT_MODEL_NAME
    )
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model.to(device)
    model.eval()

    id_to_label = {
        int(index): str(label).lower()
        for index, label in model.config.id2label.items()
    }

    rows = []
    for text in tqdm(
        output["sec_text"],
        desc="FinBERT sentiment",
    ):
        chunks = chunk_document(
            text,
            chunk_words=180,
            overlap_words=30,
            maximum_chunks=12,
        )
        probabilities = []

        for start in range(0, len(chunks), TEXT_BATCH_SIZE):
            batch = chunks[start:start + TEXT_BATCH_SIZE]
            tokens = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt",
            ).to(device)

            with torch.no_grad():
                logits = model(**tokens).logits
                batch_probabilities = torch.softmax(
                    logits, dim=-1
                ).cpu().numpy()
            probabilities.append(batch_probabilities)

        matrix = np.vstack(probabilities)
        label_columns = {
            label: matrix[:, index]
            for index, label in id_to_label.items()
        }
        positive = label_columns.get(
            "positive", np.zeros(len(matrix))
        )
        neutral = label_columns.get(
            "neutral", np.zeros(len(matrix))
        )
        negative = label_columns.get(
            "negative", np.zeros(len(matrix))
        )

        rows.append(
            {
                "finbert_positive_mean": positive.mean(),
                "finbert_neutral_mean": neutral.mean(),
                "finbert_negative_mean": negative.mean(),
                "finbert_negative_max": negative.max(),
                "finbert_negative_std": negative.std(),
                "finbert_positive_minus_negative": (
                    positive.mean() - negative.mean()
                ),
            }
        )

    sentiment_frame = pd.DataFrame(
        rows,
        index=output.index,
    )
    for feature in feature_names:
        output[feature] = sentiment_frame[feature]

    return output, feature_names


if TEXT_MODELS_READY:
    text_model_data, finbert_feature_columns = (
        build_finbert_sentiment_features(text_model_data)
    )
else:
    finbert_feature_columns = []

print("FinBERT features included:", finbert_feature_columns)

In [ ]:
def make_tfidf_pipeline(
    C: float,
    class_weight: str | None,
    minimum_document_frequency: int,
) -> Pipeline:
    return Pipeline(
        [
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    stop_words="english",
                    ngram_range=(1, 2),
                    min_df=minimum_document_frequency,
                    max_df=0.98,
                    max_features=20_000,
                    sublinear_tf=True,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    penalty="l2",
                    C=C,
                    class_weight=class_weight,
                    solver="liblinear",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def make_multimodal_pipeline(
    C: float,
    class_weight: str | None,
    include_financial: bool,
    include_embeddings: bool,
    include_finbert: bool,
    train_rows: int,
) -> Pipeline:
    transformers = []

    if include_financial and numeric_features:
        transformers.append(
            (
                "financial_numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            )
        )

    if include_financial and categorical_features:
        transformers.append(
            (
                "financial_categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(strategy="most_frequent"),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical_features,
            )
        )

    if include_embeddings:
        pca_components = max(
            1,
            min(
                TEXT_PCA_COMPONENTS,
                len(text_embedding_columns),
                train_rows - 1,
            ),
        )
        transformers.append(
            (
                "sentence_embeddings",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value=0.0,
                            ),
                        ),
                        ("scaler", StandardScaler()),
                        (
                            "pca",
                            PCA(
                                n_components=pca_components,
                                random_state=RANDOM_STATE,
                            ),
                        ),
                    ]
                ),
                text_embedding_columns,
            )
        )

    if include_finbert and finbert_feature_columns:
        transformers.append(
            (
                "finbert_sentiment",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(strategy="median"),
                        ),
                        ("scaler", StandardScaler()),
                    ]
                ),
                finbert_feature_columns,
            )
        )

    if not transformers:
        raise ValueError("No feature branch was selected.")

    preprocessor = ColumnTransformer(
        transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

    return Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "classifier",
                LogisticRegression(
                    penalty="l2",
                    C=C,
                    class_weight=class_weight,
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def choose_threshold(
    y_true: pd.Series,
    probabilities: np.ndarray,
) -> tuple[float, float]:
    scores = [
        balanced_accuracy_score(
            y_true,
            (probabilities >= threshold).astype(int),
        )
        for threshold in THRESHOLD_GRID
    ]
    best_index = int(np.argmax(scores))
    return (
        float(THRESHOLD_GRID[best_index]),
        float(scores[best_index]),
    )


def select_probability_model(
    model_name: str,
    builder,
    X_train,
    y_train: pd.Series,
    X_validation,
    y_validation: pd.Series,
) -> dict[str, Any]:
    best: dict[str, Any] | None = None

    for class_weight in [None, "balanced"]:
        for C in C_GRID:
            pipeline = builder(C, class_weight)
            pipeline.fit(X_train, y_train)
            probability = pipeline.predict_proba(
                X_validation
            )[:, 1]
            brier = brier_score_loss(
                y_validation, probability
            )

            candidate = {
                "model": model_name,
                "C": C,
                "class_weight": class_weight,
                "validation_brier": brier,
                "validation_probability": probability,
            }

            if best is None or brier < best["validation_brier"]:
                best = candidate

    assert best is not None
    threshold, threshold_score = choose_threshold(
        y_validation,
        best["validation_probability"],
    )
    best["threshold"] = threshold
    best["validation_balanced_accuracy"] = threshold_score
    return best


if TEXT_MODELS_READY:
    text_train = text_model_data[
        text_model_data["split"] == "train"
    ].copy()
    text_validation = text_model_data[
        text_model_data["split"] == "validation"
    ].copy()
    text_test = text_model_data[
        text_model_data["split"] == "test"
    ].copy()
    text_train_validation = pd.concat(
        [text_train, text_validation],
        ignore_index=True,
    )

    y_text_train = text_train["target_clean"]
    y_text_validation = text_validation["target_clean"]
    y_text_test = text_test["target_clean"]

    tfidf_min_df = 1 if len(text_train) < 100 else 2

    model_specs = {
        "Financial only (text subset)": {
            "builder": lambda C, class_weight: make_pipeline(
                C=C,
                class_weight=class_weight,
            ),
            "train_X": text_train[feature_columns],
            "validation_X": text_validation[feature_columns],
            "train_validation_X": text_train_validation[
                feature_columns
            ],
            "test_X": text_test[feature_columns],
        },
        "TF-IDF text only": {
            "builder": lambda C, class_weight: make_tfidf_pipeline(
                C=C,
                class_weight=class_weight,
                minimum_document_frequency=tfidf_min_df,
            ),
            "train_X": text_train["sec_text"],
            "validation_X": text_validation["sec_text"],
            "train_validation_X": text_train_validation[
                "sec_text"
            ],
            "test_X": text_test["sec_text"],
        },
        "Sentence embeddings only": {
            "builder": lambda C, class_weight: make_multimodal_pipeline(
                C=C,
                class_weight=class_weight,
                include_financial=False,
                include_embeddings=True,
                include_finbert=False,
                train_rows=len(text_train),
            ),
            "train_X": text_train,
            "validation_X": text_validation,
            "train_validation_X": text_train_validation,
            "test_X": text_test,
        },
        "Financial + sentence embeddings": {
            "builder": lambda C, class_weight: make_multimodal_pipeline(
                C=C,
                class_weight=class_weight,
                include_financial=True,
                include_embeddings=True,
                include_finbert=False,
                train_rows=len(text_train),
            ),
            "train_X": text_train,
            "validation_X": text_validation,
            "train_validation_X": text_train_validation,
            "test_X": text_test,
        },
    }

    if finbert_feature_columns:
        model_specs[
            "Financial + embeddings + FinBERT"
        ] = {
            "builder": lambda C, class_weight: make_multimodal_pipeline(
                C=C,
                class_weight=class_weight,
                include_financial=True,
                include_embeddings=True,
                include_finbert=True,
                train_rows=len(text_train),
            ),
            "train_X": text_train,
            "validation_X": text_validation,
            "train_validation_X": text_train_validation,
            "test_X": text_test,
        }

    text_selection_rows = []
    text_test_rows = []
    text_final_models = {}
    text_prediction_output = text_test[
        [
            column
            for column in [
                "cik",
                "ticker",
                "company_name",
                "quarter_end",
                "feature_cutoff_date",
                "label_available_date",
                "accession_number",
                "target_clean",
                "future_growth_change",
            ]
            if column in text_test.columns
        ]
    ].copy()

    text_training_prevalence = float(
        text_train_validation["target_clean"].mean()
    )

    # Same probability baseline for the text-comparison subset.
    baseline_probabilities = np.full(
        len(text_test),
        text_training_prevalence,
    )
    baseline_predictions = (
        baseline_probabilities >= 0.5
    ).astype(int)
    text_test_rows.append(
        metric_row(
            "Prior-probability baseline (text subset)",
            y_text_test,
            baseline_probabilities,
            baseline_predictions,
            baseline_probability=text_training_prevalence,
        )
    )

    for model_name, spec in model_specs.items():
        selected = select_probability_model(
            model_name=model_name,
            builder=spec["builder"],
            X_train=spec["train_X"],
            y_train=y_text_train,
            X_validation=spec["validation_X"],
            y_validation=y_text_validation,
        )

        final_pipeline = spec["builder"](
            selected["C"],
            selected["class_weight"],
        )
        final_pipeline.fit(
            spec["train_validation_X"],
            text_train_validation["target_clean"],
        )

        test_probability = final_pipeline.predict_proba(
            spec["test_X"]
        )[:, 1]
        test_prediction = (
            test_probability >= selected["threshold"]
        ).astype(int)

        result = metric_row(
            model_name,
            y_text_test,
            test_probability,
            test_prediction,
            baseline_probability=text_training_prevalence,
        )
        result["selected_C"] = selected["C"]
        result["class_weight"] = str(
            selected["class_weight"]
        )
        result["threshold"] = selected["threshold"]
        result["validation_brier"] = selected[
            "validation_brier"
        ]
        result["validation_balanced_accuracy"] = selected[
            "validation_balanced_accuracy"
        ]

        text_selection_rows.append(
            {
                "model": model_name,
                "selected_C": selected["C"],
                "class_weight": str(
                    selected["class_weight"]
                ),
                "validation_brier": selected[
                    "validation_brier"
                ],
                "validation_balanced_accuracy": selected[
                    "validation_balanced_accuracy"
                ],
                "threshold": selected["threshold"],
            }
        )
        text_test_rows.append(result)
        text_final_models[model_name] = final_pipeline
        text_prediction_output[
            f"{model_name}_probability"
        ] = test_probability
        text_prediction_output[
            f"{model_name}_prediction"
        ] = test_prediction

    text_selection_table = pd.DataFrame(
        text_selection_rows
    ).sort_values(
        ["validation_brier", "validation_balanced_accuracy"],
        ascending=[True, False],
    )

    text_model_results = pd.DataFrame(
        text_test_rows
    ).sort_values(
        ["brier_score", "balanced_accuracy"],
        ascending=[True, False],
    )

    print("Validation model selection")
    display(text_selection_table)
    print("Untouched test comparison on identical text-available rows")
    display(text_model_results)

else:
    text_selection_table = pd.DataFrame()
    text_model_results = pd.DataFrame()
    text_prediction_output = pd.DataFrame()
    text_final_models = {}

In [ ]:
if TEXT_MODELS_READY and "TF-IDF text only" in text_final_models:
    tfidf_pipeline = text_final_models["TF-IDF text only"]
    vectorizer = tfidf_pipeline.named_steps["tfidf"]
    classifier = tfidf_pipeline.named_steps["classifier"]

    vocabulary = vectorizer.get_feature_names_out()
    coefficients = classifier.coef_[0]

    term_coefficients = pd.DataFrame(
        {
            "term": vocabulary,
            "coefficient": coefficients,
            "absolute_coefficient": np.abs(coefficients),
        }
    )

    top_acceleration_terms = term_coefficients.nlargest(
        25, "coefficient"
    )[["term", "coefficient"]]
    top_deceleration_terms = term_coefficients.nsmallest(
        25, "coefficient"
    )[["term", "coefficient"]]

    print("Terms associated with acceleration")
    display(top_acceleration_terms)
    print("Terms associated with deceleration")
    display(top_deceleration_terms)

In [ ]:
if TEXT_MODELS_READY:
    text_selection_table.to_csv(
        OUTPUT_DIR / "text_validation_model_selection.csv",
        index=False,
    )
    text_model_results.to_csv(
        OUTPUT_DIR / "text_test_model_comparison.csv",
        index=False,
    )
    text_prediction_output.to_csv(
        OUTPUT_DIR / "text_test_predictions.csv",
        index=False,
    )

    if "term_coefficients" in globals():
        term_coefficients.to_csv(
            OUTPUT_DIR / "tfidf_term_coefficients.csv",
            index=False,
        )

    best_text_model_name = text_selection_table.iloc[0][
        "model"
    ]
    joblib.dump(
        {
            "pipeline": text_final_models[
                best_text_model_name
            ],
            "model_name": best_text_model_name,
            "text_embedding_model": TEXT_EMBEDDING_MODEL,
            "neutral_band": NEUTRAL_BAND,
            "minimum_text_characters": MIN_TEXT_CHARS,
            "chunk_words": CHUNK_WORDS,
            "maximum_chunks": MAX_CHUNKS_PER_FILING,
        },
        OUTPUT_DIR / "best_financial_text_model.joblib",
    )

    print(
        "Saved text-model results and best pipeline to:",
        OUTPUT_DIR,
    )

### Financial-model test results by company

This helps identify whether the overall result is being driven by only one or two firms.

In [ ]:
company_rows = []
probability_column = f"{best_model_name}_probability"
prediction_column = f"{best_model_name}_prediction"

for ticker, frame in prediction_output.groupby("ticker"):
    y_company = frame["target_clean"]
    p_company = frame[probability_column].to_numpy()
    pred_company = frame[prediction_column].to_numpy()

    company_rows.append(
        {
            "ticker": ticker,
            "rows": len(frame),
            "acceleration_rate": y_company.mean(),
            "accuracy": accuracy_score(y_company, pred_company),
            "balanced_accuracy": (
                balanced_accuracy_score(y_company, pred_company)
                if y_company.nunique() == 2
                else np.nan
            ),
            "brier_score": brier_score_loss(
                y_company, p_company
            ),
            "roc_auc": safe_auc(y_company, p_company),
        }
    )

company_results = pd.DataFrame(company_rows).sort_values(
    ["brier_score", "rows"],
    ascending=[True, False],
)
display(company_results)

## 10. Financial-model calibration plot

A well-calibrated model should lie near the diagonal. Accuracy alone cannot show whether probabilities such as 0.70 truly occur about 70% of the time.

In [ ]:
best_probability = prediction_output[
    f"{best_model_name}_probability"
].to_numpy()

calibration_frame = pd.DataFrame(
    {
        "actual": y_test.to_numpy(),
        "probability": best_probability,
    }
)
calibration_frame["bin"] = pd.cut(
    calibration_frame["probability"],
    bins=np.linspace(0.0, 1.0, 11),
    include_lowest=True,
)

calibration_table = (
    calibration_frame.groupby("bin", observed=False)
    .agg(
        mean_probability=("probability", "mean"),
        observed_rate=("actual", "mean"),
        rows=("actual", "size"),
    )
    .dropna(subset=["mean_probability"])
    .reset_index()
)
display(calibration_table)

plt.figure(figsize=(6, 6))
plt.plot(
    calibration_table["mean_probability"],
    calibration_table["observed_rate"],
    marker="o",
    label=best_model_name,
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed acceleration rate")
plt.title("Test-set calibration")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 11. Strict leave-one-company-out future-period diagnostic

For each company, the model trains only on **other companies' train + validation rows** and predicts the held company's test rows. A 0.50 threshold is used so the held company does not influence threshold tuning.

This is a demanding diagnostic. Poor performance here means the model may depend on company-specific history rather than a broadly reusable semiconductor relationship.

In [ ]:
loco_rows = []

selected_class_weight = selected_models[
    best_model_name
]["class_weight"]
selected_C = float(selected_models[best_model_name]["C"])

for ticker in sorted(test_frame["ticker"].dropna().unique()):
    loco_train = train_validation[
        train_validation["ticker"] != ticker
    ].copy()
    loco_test = test_frame[
        test_frame["ticker"] == ticker
    ].copy()

    if loco_test.empty or loco_train["target_clean"].nunique() < 2:
        continue

    loco_pipeline = make_pipeline(
        C=selected_C,
        class_weight=selected_class_weight,
    )
    loco_pipeline.fit(
        loco_train[feature_columns],
        loco_train["target_clean"],
    )

    probabilities = loco_pipeline.predict_proba(
        loco_test[feature_columns]
    )[:, 1]
    predictions = (probabilities >= 0.50).astype(int)
    y_loco = loco_test["target_clean"]

    loco_rows.append(
        {
            "held_out_ticker": ticker,
            "train_companies": loco_train["ticker"].nunique(),
            "test_rows": len(loco_test),
            "test_classes": y_loco.nunique(),
            "accuracy": accuracy_score(y_loco, predictions),
            "balanced_accuracy": (
                balanced_accuracy_score(y_loco, predictions)
                if y_loco.nunique() == 2
                else np.nan
            ),
            "brier_score": brier_score_loss(
                y_loco, probabilities
            ),
            "roc_auc": safe_auc(y_loco, probabilities),
        }
    )

loco_results = pd.DataFrame(loco_rows)
if not loco_results.empty:
    display(
        loco_results.sort_values(
            ["brier_score", "held_out_ticker"]
        )
    )
else:
    print("Not enough rows to run leave-one-company-out evaluation.")

## 12. Financial-model coefficients and saved artifacts

Positive coefficients increase the predicted probability of acceleration; negative coefficients increase the predicted probability of deceleration. Because numeric variables are standardized, their magnitudes are more comparable.

In [ ]:
preprocessor = best_model.named_steps["preprocessor"]
classifier = best_model.named_steps["classifier"]
transformed_feature_names = preprocessor.get_feature_names_out()

coefficient_table = (
    pd.DataFrame(
        {
            "feature": transformed_feature_names,
            "coefficient": classifier.coef_[0],
            "absolute_coefficient": np.abs(classifier.coef_[0]),
            "odds_ratio_per_standard_deviation": np.exp(
                classifier.coef_[0]
            ),
        }
    )
    .sort_values("absolute_coefficient", ascending=False)
    .reset_index(drop=True)
)
display(coefficient_table.head(30))

selection_table.to_csv(
    OUTPUT_DIR / "validation_model_selection.csv",
    index=False,
)
test_results.to_csv(
    OUTPUT_DIR / "test_model_metrics.csv",
    index=False,
)
prediction_output.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False,
)
company_results.to_csv(
    OUTPUT_DIR / "test_metrics_by_company.csv",
    index=False,
)
coefficient_table.to_csv(
    OUTPUT_DIR / "best_model_coefficients.csv",
    index=False,
)
if not loco_results.empty:
    loco_results.to_csv(
        OUTPUT_DIR / "leave_one_company_out_metrics.csv",
        index=False,
    )

joblib.dump(
    {
        "pipeline": best_model,
        "model_name": best_model_name,
        "feature_columns": feature_columns,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "target": "target_clean",
        "neutral_band": NEUTRAL_BAND,
        "selected_C": selected_models[best_model_name]["C"],
        "class_weight": selected_models[
            best_model_name
        ]["class_weight"],
        "threshold": best_threshold,
        "training_prevalence": training_prevalence,
    },
    OUTPUT_DIR / "best_semiconductor_model.joblib",
)

print("Saved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

## How to interpret the result

- Compare the selected model first against the **prior-probability baseline** using Brier score and Brier Skill Score.
- `Brier Skill Score > 0` means the model improves on the constant training-prevalence forecast.
- Use balanced accuracy and per-class recall when the target classes are uneven.
- If ordinary accuracy falls but Brier score, balanced accuracy, or minority-class recall improves, the revised model can still be better.
- If the 2% neutral band leaves too few rows, use 1%; do not reduce it because of test performance.
- Do not use SMOTE for this time-indexed financial dataset.
- Keep the company identifier out of the primary model unless a separate experiment demonstrates that it improves future-period performance without destroying leave-one-company-out generalization.

### Interpreting the text experiment

- The main comparison is **Financial only (text subset)** versus
  **Financial + sentence embeddings**, because both use identical rows.
- A lower test Brier score and positive Brier Skill Score indicate useful
  probability information from the filing language.
- Treat TF-IDF terms as associations, not causal explanations.
- If text improves validation but not test performance, reduce embedding
  dimensionality, collect more companies/quarters, or restrict the text to
  stable sections such as MD&A.
- Do not fine-tune a large transformer until the simpler frozen-embedding
  baseline demonstrates value; the current semiconductor panel is likely too
  small for safe end-to-end fine-tuning.
